# Build an Agent Swarm with CrewAI and Gemini 3.5 Flash

A beginner-friendly, end-to-end guide to orchestrating multiple AI agents with **[CrewAI](https://docs.crewai.com/)** powered by **Google's Gemini 3.5 Flash** model.

By the end of this notebook you will have built a small **swarm of four cooperating agents** (Researcher → Writer → Reviewer → Coordinator) that can take a single topic, do research on it, draft a blog post, review it, and hand you a polished final article.

> **A note on the model name.** Google currently advertises its Flash family as **Gemini 3.5 Flash** on the official [DeepMind Gemini Flash page](https://deepmind.google/models/gemini/flash/) and in the official [Gemini API + CrewAI example](https://ai.google.dev/gemini-api/docs/crewai-example). The model string we will use throughout this guide is `gemini/gemini-3.5-flash` — the `gemini/` prefix is the routing prefix that [LiteLLM](https://docs.litellm.ai/docs/providers/gemini) (used under the hood by CrewAI) requires to talk to Google.


## 1. What we are building

We will build a small **content-production swarm** with the following moving parts:

| Component | Role | Responsibility |
| --- | --- | --- |
| **Researcher Agent** | Gathers facts | Searches its knowledge for up-to-date points on the topic |
| **Writer Agent** | Drafts content | Turns the research notes into a blog post draft |
| **Reviewer Agent** | Polishes output | Edits the draft for clarity, tone, and correctness |
| **Manager Agent** | Coordinates | Oversees the work and produces a final summary |
| **Gemini 3.5 Flash** | The LLM brain | Powers all four agents via the Gemini API |
| **CrewAI** | The orchestrator | Wires the agents and tasks together into a single `kickoff()` |

**Example task we will run:** &gt; *"Research and write a short blog post about the benefits of local-first AI agents."*

Everything you write below lives in standard Python — no exotic frameworks, no cloud lock-in beyond the Gemini API itself.


## 2. Architecture overview

Here is what the swarm looks like at runtime. Each box is a CrewAI `Agent`, the arrows are `Task` dependencies, and the dotted arrow shows the optional `manager_llm` delegation path used by `Process.hierarchical`.

```mermaid
flowchart LR
    U[You / User Input] -->|topic| R[Researcher Agent]
    R -->|research notes| W[Writer Agent]
    W -->|draft post| RV[Reviewer Agent]
    RV -->|polished post| M[Manager / Coordinator Agent]
    M -->|final answer| OUT([Final blog post])

    G{{Gemini 3.5 Flash}} -. powers .-> R
    G -. powers .-> W
    G -. powers .-> RV
    G -. powers .-> M
```

**Key idea.** Each agent has its own *role*, *goal*, and *backstory* but they all share the same LLM. CrewAI takes care of formatting prompts, passing outputs from one task to the next, and (optionally) letting a manager agent decide what to run next.

We will use `Process.sequential` for clarity — it runs the tasks in the order we list them. At the end of the notebook, we will show how to switch to `Process.hierarchical` if you want a manager-driven workflow.


## 3. Install dependencies

Run the cell below once. It installs:

- `crewai` and `crewai-tools` — the multi-agent framework.
- `google-genai` — Google's **modern, maintained** Python SDK (the older `google-generativeai` package was deprecated on 30 Nov 2025).
- `litellm` — the routing layer CrewAI uses to call Gemini.
- `python-dotenv` — to load your API key from a `.env` file.
- `ipykernel` — so this notebook can run as a Jupyter kernel.

If you are running this in **Google Colab** the `!pip install` prefix works as-is. If you are running locally, you can also use `pip install -r requirements.txt` from the repo root.


In [ ]:
%pip install --quiet -U \
    crewai \
    crewai-tools \
    google-genai \
    litellm \
    python-dotenv

# Print installed versions so we can verify the install worked.
import importlib.metadata as md
for pkg in ["crewai", "crewai-tools", "google-genai", "litellm", "python-dotenv"]:

    print(f"{pkg:>16}  {md.version(pkg)}")


^C
Note: you may need to restart the kernel to use updated packages.


PackageNotFoundError: No package metadata was found for crewai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mistralai 1.10.0 requires opentelemetry-exporter-otlp-proto-http<2.0.0,>=1.37.0, but you have opentelemetry-exporter-otlp-proto-http 1.34.1 which is incompatible.
mistralai 1.10.0 requires opentelemetry-semantic-conventions<0.60,>=0.59b0, but you have opentelemetry-semantic-conventions 0.55b1 which is incompatible.
reflex 0.9.2 requires click>=8.2, but you have click 8.1.8 which is incompatible.
reflex-hosting-cli 0.1.63 requires click>=8.2, but you have click 8.1.8 which is incompatible.
rembg 2.0.72 requires pillow<13.0.0,>=12.1.0, but you have pillow 11.3.0 which is incompatible.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 4. Set up API keys securely

**Never hardcode API keys in a notebook.** We will load them from a `.env` file in the project root.

### 4.1 Get a Gemini API key

1. Go to [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey).
2. Click **Create API key** and copy the value.
3. Create a `.env` file next to this notebook (or at the repo root) with the following content:

```dotenv
GEMINI_API_KEY=your-real-key-here
```

> You can also use `GOOGLE_API_KEY` — CrewAI accepts either name. The repo ships a template called `.env.example` you can copy:
>
> ```bash
> cp .env.example .env       # macOS / Linux
> copy .env.example .env     # Windows
> ```


In [ ]:
"""Load the API key from .env and sanity-check it."""

import os

from pathlib import Path

from dotenv import load_dotenv



# Look for a .env file in the current working directory and one level up.

for candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env"):

    if candidate.exists():

        load_dotenv(candidate)

        print(f"Loaded environment from: {candidate}")

        break

else:

    print("No .env file found. Falling back to OS-level environment variables.")



# CrewAI / LiteLLM will read either name; we surface both for clarity.

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if not api_key or api_key == "your-gemini-api-key-here":

    raise RuntimeError(

        "No Gemini API key found. "

        "Set GEMINI_API_KEY (or GOOGLE_API_KEY) in your .env file or shell environment. "

        "Get a free key at https://aistudio.google.com/app/apikey"

    )



# Mask the key in the printout so we don't leak it in screenshots.

masked = api_key[:4] + "..." + api_key[-4:]

print(f"Gemini API key loaded OK ({masked}, length={len(api_key)}).")


## 5. Configure Gemini Flash with CrewAI

CrewAI talks to Gemini through [LiteLLM](https://docs.litellm.ai/docs/providers/gemini), so the model name has the form `gemini/<google-model-id>`. The CrewAI docs recommend `temperature=1.0` for Gemini 3.x family models ([source](https://ai.google.dev/gemini-api/docs/crewai-example)).

We will create **one shared LLM object** and pass it to every agent. This is simpler than giving each agent its own client and is the pattern used in the official Google + CrewAI example.


In [ ]:
"""Configure the Gemini 3.5 Flash LLM once and reuse it for every agent."""

from crewai import LLM



# The 'gemini/' prefix tells LiteLLM to route to Google. The suffix

# (gemini-3.5-flash) is the actual Google model id.

GEMINI_MODEL = "gemini/gemini-3.5-flash"



gemini_llm = LLM(

    model=GEMINI_MODEL,

    api_key=os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"),

    temperature=1.0,  # Google recommends 1.0 for the Gemini 3 family.

)



print(f"LLM ready: {gemini_llm.model}")

print("Provider API: Google Gemini (via LiteLLM)")


> **Heads-up on model names.** If you see a `404 models/not-found` error later, it usually means Google renamed the model. As of this writing the canonical name is `gemini-3.5-flash`; if Google rotates it you can swap in `gemini/gemini-2.5-flash` (a stable, slightly older Flash) without any other code changes. The model name is the **only** thing you need to keep in sync.


## 6. Create the agents

Every CrewAI `Agent` is defined by a `role`, a `goal`, and a `backstory`. Think of the role as their *job title*, the goal as the *outcome they are chasing*, and the backstory as their *résumé* — it primes the LLM to behave like that kind of expert.

We will create four agents:


In [ ]:
"""Agent 1 of 4 — the Researcher."""

from crewai import Agent



researcher = Agent(

    role="Senior Research Analyst",

    goal=(

        "Find accurate, up-to-date, and well-sourced information "

        "about {topic}. Produce a short list of concrete points that "

        "a writer could turn into a blog post."

    ),

    backstory=(

        "You are a meticulous research analyst who specialises in "

        "distilling complex technology topics into a handful of clear, "

        "verifiable points. You cite specific concepts, tools, and "

        "real-world examples wherever possible."

    ),

    llm=gemini_llm,

    verbose=True,           # show the agent's reasoning while it works

    allow_delegation=False, # keeps the swarm simple and predictable

)



print(f"Created agent: {researcher.role}")


In [ ]:
"""Agent 2 of 4 — the Writer."""

writer = Agent(

    role="Blog Post Writer",

    goal=(

        "Using the researcher's notes, write a clear, engaging, and "

        "well-structured blog post of about 400 words on {topic}. "

        "Use a friendly, professional tone and short paragraphs."

    ),

    backstory=(

        "You are a seasoned tech writer who has published hundreds of "

        "posts for developer audiences. You excel at turning dry "

        "research into readable prose with a strong opening, a clear "

        "middle, and a memorable closing."

    ),

    llm=gemini_llm,

    verbose=True,

    allow_delegation=False,

)



print(f"Created agent: {writer.role}")


In [ ]:
"""Agent 3 of 4 — the Reviewer."""

reviewer = Agent(

    role="Senior Editor",

    goal=(

        "Review the writer's draft for clarity, factual accuracy, and "

        "tone. Return a polished final version of the blog post that "

        "is ready to publish."

    ),

    backstory=(

        "You are a detail-oriented senior editor with 15 years of "

        "experience polishing technology articles. You cut fluff, "

        "fix awkward sentences, and make sure every claim is supported."

    ),

    llm=gemini_llm,

    verbose=True,

    allow_delegation=False,

)



print(f"Created agent: {reviewer.role}")


In [ ]:
"""Agent 4 of 4 — the Manager / Coordinator."""

manager = Agent(

    role="Content Coordinator",

    goal=(

        "Look at the research notes, draft, and reviewed post, then "

        "produce a concise final summary for the user that includes "

        "the polished blog post and a short bullet list of the key "

        "takeaways."

    ),

    backstory=(

        "You are a calm, organised editorial lead. You make sure the "

        "final deliverable to the user is tight, well-formatted, and "

        "ready to ship."

    ),

    llm=gemini_llm,

    verbose=True,

    allow_delegation=False,

)



print(f"Created agent: {manager.role}")

print(f"\nTotal agents in the swarm: {len([researcher, writer, reviewer, manager])}")


## 7. Define tasks for each agent

A `Task` in CrewAI pairs an agent with a `description` (the prompt) and an `expected_output` (the contract for what the agent should produce). Setting `expected_output` is important — it tells the LLM *exactly* what shape the final answer should take.

We will create one task per agent, in the order they should run.


In [ ]:
"""Define one Task per agent. The `context=[...]` list wires the

    output of the previous task into the prompt of the next one."""

from crewai import Task



research_task = Task(

    description=(

        "Research the topic '{topic}'. Produce a bullet list of 5-7 "

        "concrete, verifiable points that would make a strong blog post. "

        "For each point, include a one-sentence rationale."

    ),

    expected_output=(

        "A markdown bullet list of 5-7 points, each with a one-sentence "

        "explanation. No prose before or after the list."

    ),

    agent=researcher,

)



write_task = Task(

    description=(

        "Using the research notes above, write a ~400-word blog post "

        "on '{topic}'. Structure it as: a one-line hook, a 2-3 "

        "sentence intro, 3 short body sections (one per key point), "

        "and a 1-2 sentence conclusion."

    ),

    expected_output=(

        "A complete blog post of roughly 400 words, written in markdown, "

        "with a clear hook, intro, three body sections, and a conclusion."

    ),

    agent=writer,

    context=[research_task],  # gets the research output as extra context

)



review_task = Task(

    description=(

        "Review the blog post draft. Fix any factual, clarity, or tone "

        "issues. Return the polished final version of the post — do not "

        "include a list of edits, only the cleaned-up text."

    ),

    expected_output=(

        "A polished, publish-ready blog post in markdown. No commentary, "

        "no 'changes made' notes — just the final post."

    ),

    agent=reviewer,

    context=[write_task],

)



coordinate_task = Task(

    description=(

        "Look at the full chain (research notes, draft, polished post). "

        "Produce a final answer to the user that contains: "

        "(1) the polished blog post, and "

        "(2) a 3-bullet 'Key Takeaways' summary."

    ),

    expected_output=(

        "Markdown with two sections: '## Blog Post' (the polished post) "

        "and '## Key Takeaways' (exactly 3 bullets)."

    ),

    agent=manager,

    context=[research_task, write_task, review_task],

)



tasks = [research_task, write_task, review_task, coordinate_task]

print(f"Defined {len(tasks)} tasks:")

for t in tasks:

    print(f"  - {t.agent.role:<26}  ->  {t.expected_output[:60]}...")


## 8. Build the CrewAI workflow

A `Crew` is the runtime that takes a list of agents + tasks and a `Process` type, and orchestrates them. We use `Process.sequential` for clarity — tasks run in the order we list them, and each task's output becomes context for the next one (we already wired that up via `context=[...]` above).

If you want a *true* manager-driven workflow, switch `process=Process.hierarchical` and pass `manager_llm=gemini_llm`. We show both options below.


In [ ]:
"""Assemble the agents + tasks into a Crew."""

from crewai import Crew, Process



crew = Crew(

    agents=[researcher, writer, reviewer, manager],

    tasks=tasks,

    process=Process.sequential,   # tasks run in the order listed above

    verbose=True,                 # show step-by-step agent logs

    memory=False,                 # flip to True to enable shared short-term memory

)



print("Crew assembled.")

print(f"  Agents : {[a.role for a in crew.agents]}")

print(f"  Tasks  : {len(crew.tasks)}")

print(f"  Process: {crew.process}")


## 9. Run the agent swarm on an example topic

Time to actually run the swarm! The `kickoff()` method starts the workflow. We pass the topic as an `inputs` dict — any `{topic}` placeholder in the task descriptions is filled in automatically.

**Example topic:** &gt; *"The benefits of local-first AI agents."*

Because `verbose=True` is set, you will see each agent's reasoning in real time. This may take 30-90 seconds depending on the Gemini API load.


In [ ]:
"""Kick off the swarm. The string passed via `inputs={'topic': ...}` is

    substituted into every '{topic}' placeholder in the task prompts."""

from datetime import datetime



topic = "The benefits of local-first AI agents"



print(f"Starting crew at {datetime.now().isoformat(timespec='seconds')}")

print(f"Topic: {topic!r}\n")



result = crew.kickoff(inputs={"topic": topic})



print(f"\nFinished at {datetime.now().isoformat(timespec='seconds')}")

print(f"Result type: {type(result).__name__}")


## 10. Display and explain the final output

CrewAI returns a `CrewOutput` object. The most useful attribute is `.raw`, which is the final answer string produced by the last task in the chain. The intermediate task outputs are also available on each `TaskOutput` if you want to inspect them.

In our chain the **last task** belongs to the `Manager` agent, so `.raw` contains both the polished blog post and the 3-bullet key-takeaways summary we asked for in `coordinate_task.expected_output`.


In [ ]:
"""Pretty-print the final result and expose the individual task outputs."""

from IPython.display import Markdown, display



print("=" * 70)

print("FINAL ANSWER (manager agent)")

print("=" * 70)

display(Markdown(result.raw))



print()

print("=" * 70)

print("TASK-BY-TASK TRACE")

print("=" * 70)

for i, task_out in enumerate(result.tasks_output, start=1):

    agent_name = task_out.agent if isinstance(task_out.agent, str) else task_out.agent

    print(f"\n--- Task {i}: {agent_name} ---")

    # Truncate the long middle outputs so the trace stays readable.

    preview = task_out.raw.strip().replace("\n", " ")

    print(preview[:200] + ("..." if len(preview) > 200 else ""))


## 11. Error handling and debugging tips

When something goes wrong with a CrewAI + Gemini pipeline, the error usually falls into one of four buckets. Here is a quick troubleshooting table you can come back to.

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| `RuntimeError: No Gemini API key found` | `.env` not loaded, or key still set to the placeholder | Verify `.env` exists in CWD or its parent, and the value is your real key |
| `litellm.AuthenticationError: geminiException` | API key invalid, revoked, or restricted to a different project | Create a new key at [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey) |
| `404 models/gemini-3.5-flash not found` | Google rotated the model name on the API | Try `gemini/gemini-2.5-flash` (the previous stable Flash) or check [ai.google.dev/gemini-api/docs/models](https://ai.google.dev/gemini-api/docs/models) |
| `litellm.BadRequestError: ... models/ ...` | LiteLLM is double-prefixing the model id | Make sure your `LLM(model=...)` string is `"gemini/gemini-3.5-flash"` — only one `gemini/` prefix. See [crewAI#2645](https://github.com/crewAIInc/crewAI/issues/2645) |
| Agents produce empty / very short answers | Temperature too high, or `expected_output` is too vague | Lower `temperature` to `0.3`-`0.5` and tighten the `expected_output` string |
| `ModuleNotFoundError: google.generativeai` | You installed the legacy SDK | `pip install -U google-genai` instead (the legacy package was retired 30 Nov 2025) |

**Two debug toggles that save hours:**

```python
import os
os.environ["LITELLM_LOG"] = "DEBUG"     # full request/response logging
import litellm; litellm._turn_on_debug() # alternative, even more verbose
```

Adding either of these before `crew.kickoff()` will print the exact request CrewAI sends to Google, which makes 90% of the remaining issues obvious.


In [ ]:
"""Robust wrapper around crew.kickoff() — drop-in replacement.

    Demonstrates the standard error-handling pattern for a real app."""

import litellm



def run_crew_safely(crew, inputs, debug: bool = False):

    """Run a Crew and return its result, with friendly error messages."""

    if debug:

        os.environ["LITELLM_LOG"] = "DEBUG"

        litellm._turn_on_debug()

    try:

        return crew.kickoff(inputs=inputs)

    except litellm.AuthenticationError as e:

        raise RuntimeError(

            "Gemini rejected the API key. Check that GEMINI_API_KEY (or "

            "GOOGLE_API_KEY) in your .env is correct and active."

        ) from e

    except litellm.NotFoundError as e:

        raise RuntimeError(

            f"The Gemini model '{GEMINI_MODEL}' was not found. "

            f"Google may have renamed it; try 'gemini/gemini-2.5-flash'."

        ) from e

    except litellm.RateLimitError as e:

        raise RuntimeError(

            "You hit a Gemini rate limit. Wait a minute and try again, "

            "or reduce the number of tasks / agents in the crew."

        ) from e



print("Helper defined. You can now call `run_crew_safely(crew, {'topic': '...'})`.")


## 12. Optional improvements

The basic swarm works, but there are a handful of small upgrades that make it dramatically more useful in real projects. Pick the ones that match your goal.


### 12.1 Save the final post to a file

Most of the time you want the final output to land on disk so you can publish, archive, or post-process it. This snippet writes the polished post to `output/<slug>.md`.


In [ ]:
"""Save the final post to a markdown file."""

import re

from pathlib import Path



output_dir = Path("output")

output_dir.mkdir(exist_ok=True)



# Turn 'The benefits of local-first AI agents' into a filename-safe slug.

slug = re.sub(r"[^a-z0-9]+", "-", topic.lower()).strip("-")

out_path = output_dir / f"{slug}.md"



out_path.write_text(result.raw, encoding="utf-8")

print(f"Saved {len(result.raw):,} characters to {out_path.resolve()}")


### 12.2 Add a tool (e.g. a web search)

In this notebook the Researcher relies entirely on what Gemini already knows. If you want it to fetch *fresh* information, give it a tool. CrewAI ships a [SerperDevTool](https://docs.crewai.com/tools/serperdevtool) that needs only a `SERPER_API_KEY` in your `.env`.

```python
from crewai_tools import SerperDevTool

researcher.tools = [SerperDevTool()]   # now the researcher can search the web
```

Other handy tools you can mix in the same way: `WebsiteSearchTool`, `FileReadTool`, `ScrapeWebsiteTool`, and `CodeInterpreterTool`. See the [CrewAI tools catalogue](https://docs.crewai.com/en/concepts/tools).


### 12.3 Enable shared memory

Set `memory=True` on the `Crew` to let agents remember previous task outputs and the conversation history. This is what allows the writer to build on the researcher's exact phrasing, and the reviewer to focus on deltas rather than re-reading the entire draft.

```python
crew = Crew(agents=[...], tasks=[...], process=Process.sequential, memory=True)
```

By default, memory is short-term only. To persist memory across runs, set `memory=True` *and* configure a storage backend (Chroma, SQLite, etc.) — see the [memory docs](https://docs.crewai.com/en/concepts/memory).


### 12.4 Switch to a hierarchical (manager-driven) process

Right now we use `Process.sequential` — tasks run in a fixed order. If you want the Manager agent to dynamically decide what should run next (and re-delegate on failure), switch to `Process.hierarchical` and point the crew at a `manager_llm`:

```python
crew = Crew(
    agents=[researcher, writer, reviewer, manager],
    tasks=tasks,
    process=Process.hierarchical,
    manager_llm=gemini_llm,    # <-- required for hierarchical
    verbose=True,
)
```

Hierarchical mode is more flexible but slower and more expensive (every delegation is another LLM call). It shines for open-ended research tasks; for a known 4-step pipeline like ours, sequential is usually the right choice.


### 12.5 Change the model per agent

Different agents benefit from different models. You can pass a different `LLM(...)` object to each agent — e.g. use a faster/cheaper Flash for the writer and reviewer, and a stronger Pro model for the researcher if you have access to one:

```python
from crewai import LLM

fast_llm = LLM(model="gemini/gemini-3.5-flash", temperature=1.0)
strong_llm = LLM(model="gemini/gemini-2.5-pro", temperature=0.4)

researcher.llm = strong_llm   # deep research
writer.llm     = fast_llm     # fast drafting
reviewer.llm   = fast_llm     # fast editing
manager.llm    = fast_llm     # fast coordination
```


## 13. Conclusion & next steps

You have just built a complete **4-agent CrewAI swarm** powered by **Gemini 3.5 Flash**:

1. A *Researcher* that pulls together the key points on a topic.
2. A *Writer* that turns those points into a blog post.
3. A *Reviewer* that polishes the draft.
4. A *Manager* that delivers the final, formatted answer to you.

From here, the natural next steps are:

- **Add real tools** (web search, file reading, code execution) so the researcher can pull in *live* data instead of relying on the LLM's training cutoff.
- **Persist memory** with Chroma or SQLite so each run can build on the previous one.
- **Tighten the prompts** — the quality of a CrewAI swarm is dominated by the quality of the `role`, `goal`, `backstory`, and `expected_output` strings you write.
- **Swap the LLM** — change one line (`GEMINI_MODEL`) to point the whole crew at `gemini-2.5-pro`, an OpenAI model, a local Ollama model, etc.
- **Wrap it in a Flow** — for production, use [CrewAI Flows](https://docs.crewai.com/en/concepts/flows) to add state, retries, and human-in-the-loop checkpoints.

Happy swarming!
